# Calling a C library with `ctypes`

This notebook calls from Python the functions of a small C library, written for you: you only write Python.

1. Loading the library and declaring the types
2. Passing a NumPy array to a C function
3. Getting back an array filled by C
4. Faster than Python?
5. Going further: the cost of a call

The steps build on each other: each exercise uses what the previous ones' solutions define. Each exercise has a cell for your code, then additional hints and its solution, collapsed: open them if needed.

The notebook compiles C with `gcc`, installed in Colab as in the Codespace.

## The C library

The next cell writes the file `signal.c`, which defines three functions:

- `interpolate(a, b, t)` returns the point at the fraction `t` of the way from `a` to `b`;
- `mean(x, n)` returns the mean of the `n` values of the array `x`;
- `smooth(x, y, n, k)` writes into `y` the moving averages of `x` over windows of `k` consecutive values, that is `n - k + 1` averages, and returns `0`, or `-1` if `k` isn't between `1` and `n`. From one window to the next, it adds the value that comes in and removes the one that goes out.

In [ ]:
%%writefile signal.c
#include <math.h>

double interpolate(double a, double b, double t) {
  return a + (b - a) * t;
}

double mean(const double *x, int n) {
  if (n <= 0) {
    return NAN;
  }
  double total = 0.0;
  for (int i = 0; i < n; i++) {
    total += x[i];
  }
  return total / n;
}

int smooth(const double *x, double *y, int n, int k) {
  if (k <= 0 || k > n) {
    return -1;
  }
  double total = 0.0;
  for (int i = 0; i < k; i++) {
    total += x[i];
  }
  y[0] = total / k;
  for (int i = k; i < n; i++) {
    total += x[i] - x[i - k];
    y[i - k + 1] = total / k;
  }
  return 0;
}

The one after compiles it into a shared library, `libsignal.so`: `-shared` makes a library rather than an executable, `-fPIC` code that works at whatever address it is loaded.

In [ ]:
!gcc -O2 -Wall -fPIC -shared signal.c -o libsignal.so

## 1. Loading the library and declaring the types

The next cell loads `libsignal.so` with `ctypes.CDLL` and calls `interpolate` without declaring anything: `ctypes` doesn't know how to convert a Python `float`, and the call fails.

In [ ]:
import ctypes

import numpy as np

lib = ctypes.CDLL("./libsignal.so")

try:
  lib.interpolate(2.0, 4.0, 0.25)
except ctypes.ArgumentError as error:
  print("Error:", error)

Declare the types of the arguments (`argtypes`) and of the result (`restype`) of `lib.interpolate`, from its C signature, then print `lib.interpolate(2.0, 4.0, 0.25)` and `lib.interpolate(10, 20, 0.5)`. You should get `2.5` and `15.0`: once the types are declared, `ctypes` converts integers too.

In [ ]:
# Your code here

### Additional hints

`a`, `b` and `t` are C `double`s, that is `ctypes.c_double`, as is the result. `argtypes` is a list, one type per argument, in order; `restype` is a single type.

Without `restype`, `ctypes` reads the result as a C `int`: try it, the call returns an integer unrelated to `2.5`.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
lib.interpolate.argtypes = [...]
lib.interpolate.restype = ...
print(lib.interpolate(2.0, 4.0, 0.25))
print(lib.interpolate(10, 20, 0.5))
```

### Solution

In [ ]:
lib.interpolate.argtypes = [ctypes.c_double, ctypes.c_double, ctypes.c_double]
lib.interpolate.restype = ctypes.c_double
print(lib.interpolate(2.0, 4.0, 0.25))
print(lib.interpolate(10, 20, 0.5))

## 2. Passing a NumPy array to a C function

`mean` gets a pointer to the first element of an array of `double`s and its length. Declare its types, then write a Python function `mean(values)` that accepts a list or a NumPy array, converts it to a contiguous NumPy array of `float64` and calls `lib.mean` with a pointer to its data. Print `mean([1, 2, 3, 4])`, which is `2.5`, then `mean(measures)` and `np.mean(measures)`, which must agree. What does `mean([])` return?

**Useful documentation**

- https://numpy.org/doc/stable/reference/generated/numpy.ascontiguousarray.html
- https://numpy.org/doc/stable/reference/generated/numpy.ndarray.ctypes.html

In [ ]:
rng = np.random.default_rng(0)
t = np.linspace(0, 1, 1_000)
measures = np.sin(2 * np.pi * 3 * t) + rng.normal(0, 0.3, t.size)

# Your code here

### Additional hints

- `ctypes.POINTER(ctypes.c_double)` is the type "pointer to a `double`", `ctypes.c_int` the one of `n`, and the result is a `ctypes.c_double`.
- `np.ascontiguousarray(values, dtype=np.float64)` returns an array whose values are C `double`s stored side by side in memory, which is what C expects (without a copy if that is already the case).
- `array.ctypes.data_as(ctypes.POINTER(ctypes.c_double))` gives the pointer to its data.

`mean([])` returns `nan`: the C code returns `NAN` when `n` is 0, as `np.mean` does.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
lib.mean.argtypes = [..., ...]
lib.mean.restype = ...


def mean(values) -> float:
  array = np.ascontiguousarray(...)
  pointer = array.ctypes.data_as(...)
  return lib.mean(pointer, ...)


print(mean([1, 2, 3, 4]))
print(mean(measures), np.mean(measures))
print(mean([]))
```

### Solution

In [ ]:
lib.mean.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.c_int]
lib.mean.restype = ctypes.c_double


def mean(values) -> float:
  array = np.ascontiguousarray(values, dtype=np.float64)
  pointer = array.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
  return lib.mean(pointer, len(array))


print(mean([1, 2, 3, 4]))
print(mean(measures), np.mean(measures))
print(mean([]))

## 3. Getting back an array filled by C

`smooth` writes its result into an array `y` the caller gives it. This time, declare its two arrays with `np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS")`: `ctypes` then accepts a NumPy array as is, and checks its type. Write `smooth(values, k)`, which returns a new NumPy array of the `len(values) - k + 1` moving averages and raises a `ValueError` when the C code returns `-1`. Check with `np.allclose` that `smooth(measures, 5)` gives the same result as `np.convolve(measures, np.ones(5) / 5, mode="valid")`, then that `smooth(measures, 0)` does raise a `ValueError`.

**Useful documentation**

- https://numpy.org/doc/stable/reference/routines.ctypeslib.html#numpy.ctypeslib.ndpointer
- https://numpy.org/doc/stable/reference/generated/numpy.empty_like.html
- https://numpy.org/doc/stable/reference/generated/numpy.allclose.html

In [ ]:
# Your code here

### Additional hints

- The types: `[array_double, array_double, ctypes.c_int, ctypes.c_int]`, where `array_double` is the type `np.ctypeslib.ndpointer` returns, and `ctypes.c_int` for the result.
- `smooth` still converts its argument with `np.ascontiguousarray`: the `ndpointer` type refuses a list (`ctypes.ArgumentError`) and an array of integers.
- `np.empty_like(x)` allocates `y`, as long as `x`, without initializing it: the C code fills its first `len(x) - k + 1` cells, which `smooth` returns.

Example: `smooth([1, 2, 3, 4, 5], 2)` returns `array([1.5, 2.5, 3.5, 4.5])`.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
array_double = np.ctypeslib.ndpointer(dtype=..., flags="C_CONTIGUOUS")
lib.smooth.argtypes = [...]
lib.smooth.restype = ...


def smooth(values, k: int) -> np.ndarray:
  x = np.ascontiguousarray(values, dtype=np.float64)
  y = np.empty_like(x)
  if ... != 0:
    msg = f"invalid window: {k}"
    raise ValueError(msg)
  return y[...]


print(smooth([1, 2, 3, 4, 5], 2))
reference = np.convolve(measures, np.ones(5) / 5, mode="valid")
print(np.allclose(..., reference))
try:
  smooth(measures, 0)
except ValueError as error:
  print("Error:", error)
```

### Solution

In [ ]:
array_double = np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS")
lib.smooth.argtypes = [array_double, array_double, ctypes.c_int, ctypes.c_int]
lib.smooth.restype = ctypes.c_int


def smooth(values, k: int) -> np.ndarray:
  x = np.ascontiguousarray(values, dtype=np.float64)
  y = np.empty_like(x)
  if lib.smooth(x, y, len(x), k) != 0:
    msg = f"invalid window: {k}"
    raise ValueError(msg)
  return y[: len(x) - k + 1]


print(smooth([1, 2, 3, 4, 5], 2))
reference = np.convolve(measures, np.ones(5) / 5, mode="valid")
print(np.allclose(smooth(measures, 5), reference))
try:
  smooth(measures, 0)
except ValueError as error:
  print("Error:", error)

## 4. Faster than Python?

The next cell defines `smooth_python`, the same computation in pure Python, and a signal of a million values, `long_signal`, and its copy as a Python list, `long_list`. Time with `%timeit` `smooth_python` on `long_list`, then your `smooth` and `np.convolve` (as in 3) on `long_signal`, with windows of 50 values. Which one is the fastest? Why is `np.convolve`, written in C too, slower than `smooth`?

**Useful documentation**

- https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-timeit

In [ ]:
def smooth_python(values: list[float], k: int) -> list[float]:
  total = sum(values[:k])
  result = [total / k]
  for i in range(k, len(values)):
    total += values[i] - values[i - k]
    result.append(total / k)
  return result


long_signal = rng.normal(size=1_000_000)
long_list = long_signal.tolist()

# Your code here

### Additional hints

`%timeit` followed by an expression runs it several times and prints its mean time. `smooth_python` gets a list: a Python loop over a NumPy array is even slower, each element read becoming a Python object.

For `np.convolve`, count the operations: how many multiplications for one average, and how many operations does `smooth` make?

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
%timeit smooth_python(long_list, 50)
%timeit smooth(...)
%timeit np.convolve(...)
```

### Solution

In [ ]:
%timeit smooth_python(long_list, 50)
%timeit smooth(long_signal, 50)
%timeit np.convolve(long_signal, np.ones(50) / 50, mode="valid")
# C is tens of times faster than the Python loop. np.convolve, compiled too,
# makes 50 multiplications per average, where smooth adds one value and
# removes another: the algorithm makes the difference.

## 5. Going further: the cost of a call

Each call of a C function through `ctypes` converts its arguments and its result. Time with `%timeit` the computation of the list of `lib.interpolate(0.0, 1.0, t)` for each `t` of `positions` (100,000 values), then that of the same list in Python, `0.0 + (1.0 - 0.0) * t`. What do you conclude about how to share the work between Python and C?

In [ ]:
positions = np.linspace(0, 1, 100_000).tolist()

# Your code here

### Additional hints

A list comprehension, `[... for t in positions]`, after `%timeit`.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
%timeit [lib.interpolate(0.0, 1.0, t) for t in positions]
%timeit [... for t in positions]
```

### Solution

In [ ]:
%timeit [lib.interpolate(0.0, 1.0, t) for t in positions]
%timeit [0.0 + (1.0 - 0.0) * t for t in positions]
# Calling C is more than ten times slower than computing in Python: converting
# the arguments and the result costs much more than the computation. So C is
# given a whole array in one call, as smooth is, not one value at a time.